# Benchmark Go (gograph + vLLM)

Notebook parametrizável para rodar `base-ctx-imports`, `base-ctx-fix`, `ft-ctx` e `ft-ctx-fix` no Colab com **vLLM**, **Go 1.27.0**, **disco local `/content/work/`**, **`nproc` workers** e **retomada por função no Drive**.


In [ ]:
# Célula 1: Verificar GPU, nproc e definir dtype fixo
!nvidia-smi
import os, torch
assert torch.cuda.is_available(), "GPU não disponível!"
gpu_name = torch.cuda.get_device_name(0)
cc = torch.cuda.get_device_capability(0)
DTYPE = "bfloat16" if cc[0] >= 8 else "float16"
NPROC = os.cpu_count() or 2
print(f"GPU: {gpu_name} | CC: {cc} | DTYPE: {DTYPE} | nproc: {NPROC}")


In [ ]:
# Célula 2: Montar Drive e copiar (rsync) repos/, db/ e modcache/ para /content/work/
from google.colab import drive
drive.mount("/content/drive")

import os, time, subprocess

DRIVE_ROOT = "/content/drive/MyDrive/gograph"
WORK_ROOT = "/content/work"
os.makedirs(f"{DRIVE_ROOT}/bench", exist_ok=True)
for d in ["repos", "db", "modcache", "gocache"]:
    os.makedirs(f"{WORK_ROOT}/{d}", exist_ok=True)

t0 = time.time()
for d in ["repos", "db", "modcache"]:
    src = f"{DRIVE_ROOT}/{d}/"
    dst = f"{WORK_ROOT}/{d}/"
    if os.path.exists(src):
        subprocess.run(["rsync", "-a", "--delete", src, dst], check=True)
copy_s = time.time() - t0
print(f"Cópia Drive -> /content/work concluída em {copy_s:.1f}s")


In [ ]:
# Célula 3: Instalar vLLM mais recente compatível com o Colab e registrar versões finais
!pip install -q -U vllm transformers peft accelerate requests
import json, torch, vllm, transformers, peft, accelerate
VERSIONS = {
    "vllm": vllm.__version__,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "accelerate": accelerate.__version__,
}
print("Versões finais (fixas para todas as execuções):", json.dumps(VERSIONS, indent=2))
with open(f"{DRIVE_ROOT}/bench/versions.json", "w") as f:
    json.dump(VERSIONS, f, indent=2)


In [ ]:
# Célula 4: Instalar Go 1.27.0 (mesma versão local) e compilar gograph
%%bash
set -e
if [ ! -d /usr/local/go ] || ! /usr/local/go/bin/go version | grep -q "go1.27.0"; then
  wget -q https://go.dev/dl/go1.27.0.linux-amd64.tar.gz -O /tmp/go.tar.gz
  rm -rf /usr/local/go && tar -C /usr/local -xzf /tmp/go.tar.gz
fi
export GOROOT=/usr/local/go
export PATH=$GOROOT/bin:$PATH
go version

if [ ! -d /content/gograph ]; then
  echo "ATENÇÃO: copie/clone o repositório gograph para /content/gograph antes de compilar."
else
  cd /content/gograph && go build -o /usr/local/bin/gograph ./cmd/gograph
fi


In [ ]:
# Célula 5: Garantir go mod download em /content/work/repos (usando cache local)
import subprocess, os

REPOS = [
    "gorilla__mux",
    "charmbracelet__bubbletea",
    "docker__compose",
    "nektos__act",
    "FiloSottile__mkcert",
    "qax-os__excelize",
]

env = os.environ.copy()
env["GOROOT"] = "/usr/local/go"
env["PATH"] = "/usr/local/go/bin:" + env["PATH"]
env["GOMODCACHE"] = f"{WORK_ROOT}/modcache"
env["GOCACHE"] = f"{WORK_ROOT}/gocache"
env["GOTOOLCHAIN"] = "local"

for r in REPOS:
    repo_dir = f"{WORK_ROOT}/repos/{r}"
    if os.path.exists(repo_dir):
        subprocess.run(["go", "mod", "download"], cwd=repo_dir, env=env, check=False)
print("Módulos Go prontos em /content/work")


In [ ]:
# Célula 6: Parâmetros FIXOS de geração (iguais para base e ft, com e sem fix)
import json

BASE_MODEL = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
GEN_CONFIG = {
    "base_model": BASE_MODEL,
    "dtype": DTYPE,
    "versions": VERSIONS,
    "temperature": 0.0,
    "top_p": 1.0,
    "repetition_penalty": 1.05,
    "max_tokens": 1000,
    "seed": 42,
    "workers": NPROC,
}

# Opcional: adapters LoRA para --enable-lora (ex: {"lr-5e-5": f"{DRIVE_ROOT}/out/lr-5e-5"})
LORA_ADAPTERS = {}

MODES = [
    {"prefix": "val-vllm-base-ctx-imports", "model": BASE_MODEL, "retries": 0, "vet": False, "tail": True},
    {"prefix": "vllm-base-ctx-imports",     "model": BASE_MODEL, "retries": 0, "vet": False, "tail": False},
    {"prefix": "val-vllm-base-ctx-fix",     "model": BASE_MODEL, "retries": 2, "vet": True,  "tail": True},
    {"prefix": "vllm-base-ctx-fix",         "model": BASE_MODEL, "retries": 2, "vet": True,  "tail": False},
]
print(json.dumps(GEN_CONFIG, indent=2))


In [ ]:
# Célula 7: Subir vLLM + proxy Ollama (/api/chat -> /v1/chat/completions) + teste de verificação
import subprocess, time, threading, requests, json
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

lora_args = []
if LORA_ADAPTERS:
    lora_args += ["--enable-lora", "--max-lora-rank", "32", "--lora-modules"]
    lora_args += [f"{name}={path}" for name, path in LORA_ADAPTERS.items()]

vllm_cmd = [
    "python3", "-m", "vllm.entrypoints.openai.api_server",
    "--model", BASE_MODEL,
    "--dtype", DTYPE,
    "--max-model-len", "8192",
    "--port", "8000",
] + lora_args

vllm_proc = subprocess.Popen(vllm_cmd, stdout=open(f"{DRIVE_ROOT}/bench/vllm.log", "w"), stderr=subprocess.STDOUT)
for _ in range(90):
    try:
        if requests.get("http://127.0.0.1:8000/v1/models", timeout=2).status_code == 200:
            print("vLLM pronto!")
            break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError(f"vLLM não iniciou; veja {DRIVE_ROOT}/bench/vllm.log")

LOG_FIRST_PAYLOAD = True

class OllamaProxy(BaseHTTPRequestHandler):
    def log_message(self, *args): pass
    def do_POST(self):
        global LOG_FIRST_PAYLOAD
        if self.path != "/api/chat":
            self.send_error(404); return
        length = int(self.headers.get("Content-Length", 0))
        req = json.loads(self.rfile.read(length))
        opts = req.get("options", {})
        payload = {
            "model": req.get("model", BASE_MODEL),
            "messages": req.get("messages", []),
            "temperature": opts.get("temperature", GEN_CONFIG["temperature"]),
            "top_p": opts.get("top_p", GEN_CONFIG["top_p"]),
            "repetition_penalty": opts.get("repeat_penalty", GEN_CONFIG["repetition_penalty"]),
            "max_tokens": opts.get("num_predict", GEN_CONFIG["max_tokens"]),
            "seed": opts.get("seed", GEN_CONFIG["seed"]),
        }
        if LOG_FIRST_PAYLOAD:
            LOG_FIRST_PAYLOAD = False
            dbg = {k: v for k, v in payload.items() if k != "messages"}
            print("Payload enviado ao vLLM (1a chamada):", json.dumps(dbg))
        r = requests.post("http://127.0.0.1:8000/v1/chat/completions", json=payload, timeout=300)
        if r.status_code != 200:
            self.send_response(r.status_code); self.end_headers(); self.wfile.write(r.content); return
        data = r.json()
        choice = data["choices"][0]
        content = choice["message"]["content"]
        finish_reason = choice.get("finish_reason", "stop")
        out = json.dumps({"message": {"content": content}, "done_reason": finish_reason}).encode()
        self.send_response(200)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(out)

server = ThreadingHTTPServer(("127.0.0.1", 11434), OllamaProxy)
threading.Thread(target=server.serve_forever, daemon=True).start()

# Teste de verificação: confirmar parâmetros e finish_reason == "length" -> done_reason == "length"
test_resp = requests.post("http://127.0.0.1:11434/api/chat", json={
    "model": BASE_MODEL,
    "messages": [{"role": "user", "content": "Write a long Go HTTP server with 20 handlers."}],
    "options": {"temperature": 0, "top_p": 1, "repeat_penalty": 1.05, "num_predict": 5, "seed": 42}
}, timeout=30).json()
print("Teste proxy done_reason:", test_resp.get("done_reason"))
assert test_resp.get("done_reason") == "length", f"Esperado done_reason=length, veio {test_resp}"
print("Proxy verificado com sucesso!")


In [ ]:
# Célula 8: Executar modos configurados em /content/work (retomável por função no Drive)
import subprocess, os, time, json, datetime

env = os.environ.copy()
env["GOROOT"] = "/usr/local/go"
env["PATH"] = "/usr/local/go/bin:" + env["PATH"]
env["GOMODCACHE"] = f"{WORK_ROOT}/modcache"
env["GOCACHE"] = f"{WORK_ROOT}/gocache"

for m in MODES:
    prefix = m["prefix"]
    cfg_log = {**GEN_CONFIG, **m, "gpu": gpu_name, "started_at": datetime.datetime.now().isoformat()}
    with open(f"{DRIVE_ROOT}/bench/{prefix}_config.json", "w") as cf:
        json.dump(cfg_log, cf, indent=2)
    print(f"\n========== {prefix} (workers={NPROC}) ==========")
    t_mode = time.time()
    total_fns = 0
    for r in REPOS:
        out_file = f"{DRIVE_ROOT}/bench/{prefix}-{r}.jsonl"
        repo_dir = f"{WORK_ROOT}/repos/{r}"
        db_file = f"{WORK_ROOT}/db/{r}.db"
        if not os.path.exists(repo_dir) or not os.path.exists(db_file):
            print(f"Pulando {r} (repo ou db ausente)")
            continue
        if m["tail"] and r in ("gorilla__mux", "FiloSottile__mkcert"):
            open(out_file, "a").close()
            continue
        cmd = [
            "gograph", "bench",
            "-ctx", "real", "-budget", "24000",
            "-db", db_file, "-go", "/usr/local/go/bin/go",
            "-model", m["model"], "-ollama-host", "http://127.0.0.1:11434",
            "-n", "30" if m["tail"] else "60",
            "-workers", str(NPROC),
            "-tests", "-goimports",
            "-retries", str(m["retries"]),
            "-out", out_file,
        ]
        if m["vet"]:
            cmd.append("-vet")
        if m["tail"]:
            cmd.append("-tail")
        cmd.append(repo_dir)
        print("Rodando:", r)
        subprocess.run(cmd, env=env, check=True)
        total_fns += sum(1 for _ in open(out_file))
    dt = time.time() - t_mode
    print(f"Modo {prefix} concluído em {dt/60:.1f} min ({dt/max(total_fns,1):.2f}s/fn em média)")


In [ ]:
# Célula 9: Verificação Ollama vs vLLM + Comparação vllm-base-ctx-imports vs vllm-base-ctx-fix
!echo "=== VERIFICAÇÃO: Ollama vs vLLM (TESTE 324) ==="
!python3 /content/gograph/bench/comparar.py --dir /content/drive/MyDrive/gograph/bench --audit-dir /content/drive/MyDrive/gograph/bench base-ctx-imports vllm-base-ctx-imports

!echo ""
!echo "=== ETAPA 3: VALIDAÇÃO (120) ==="
!python3 /content/gograph/bench/comparar.py --dir /content/drive/MyDrive/gograph/bench --audit-dir /content/drive/MyDrive/gograph/bench val-vllm-base-ctx-imports val-vllm-base-ctx-fix

!echo ""
!echo "=== ETAPA 3: TESTE (324) ==="
!python3 /content/gograph/bench/comparar.py --dir /content/drive/MyDrive/gograph/bench --audit-dir /content/drive/MyDrive/gograph/bench vllm-base-ctx-imports vllm-base-ctx-fix
